<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/NBM_Deterministic_Percentile_Picker_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ⚙️ Imports (Run only 1 time per session)
!pip install eccodes==2.38.3 # need this version to avoid a google colab crash
!pip install cfgrib
!pip install pygrib
!pip install xarray
!pip install cartopy
import os, re, sys
import datetime as dt
import requests
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import pandas as pd
import cfgrib
import pygrib
try:
    import numpy as np
    import xarray as xr
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    from PIL import Image
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.patches import Patch
    from matplotlib.colors import Normalize, LinearSegmentedColormap, BoundaryNorm, TwoSlopeNorm, to_hex
except ImportError:
    raise ImportError("herbie.paint requires matplotlib.")

In [ ]:
#@title ⚙️ NBM EXP Wind Subset Downloader
#@markdown **Select your NBM initial date**
init_date ="2025-10-27" #@param {"type":"date"}
#@markdown **Select your run time**
init_time = 13 #@param {"type": "slider", min:1, max:19, step:6}

#@markdown **Select your valid date**
valid_date ="2025-10-30" #@param {"type":"date"}
#@markdown **Select your valid time**
valid_time = 0 #@param {"type": "slider", min:0, max:240, step:6}

BASE = "https://noaa-nbm-para-pds.s3.amazonaws.com"

percentiles = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]

def build_init_datetime(date_str: str, time_str: str) -> dt.datetime:
  return dt.datetime.strptime(init_date + " " + str(init_time), "%Y-%m-%d %H")

def build_valid_datetime(date_str: str, time_str: str) -> dt.datetime:
  return dt.datetime.strptime(valid_date + " " + str(valid_time), "%Y-%m-%d %H")

def build_forecast_projection(init_date_obj: dt.datetime, fcst_date_obj: dt.datetime):
  return int((fcst_date_obj-init_date_obj).total_seconds()/3600)

def build_qmd_initdatetime(date_obj: dt.datetime) -> dt.datetime:
  return date_obj-dt.timedelta(hours=7)

def build_url(date_obj: dt.date, hour: int, projection: int, product: str = "core", region: str = "ak") -> str:
    ymd = date_obj.strftime("%Y%m%d")
    hh = f"{int(hour):02d}"
    fname = f"blend.t{hh}z.{product}.f{projection:03d}.{region}.grib2"
    return f"{BASE}/blend.{ymd}/{hh}/{product}/{fname}"

def build_search_strings_core(element: str, hour: int):
    if element.upper() == "WIND":
        # Note the colon after WIND and no space before the hour number.
        return [f":{element}:10 m above ground:{hour} hour fcst"]
    return []

def build_search_string_qmd(element: str, hour: int, percentiles: list[int]):
    if element.upper() == "WIND":
        # Same colon pattern; QMD percentiles are listed as e.g. "50% level"
        return [f":{element}:10 m above ground:{hour} hour fcst:{perc}% level"
                for perc in percentiles]
    return []


def ensure_parent_dir(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

def download_subset(remote_url: str,
                    search_strings,
                    local_filename: str,
                    exclude_phrase: str = None,
                    require_all_matches: bool = True,
                    timeout: int = 30) -> str | None:
    """Subset a GRIB2 using byte ranges from its .idx file."""
    remote_file = os.path.basename(remote_url)
    idx_url = remote_url + ".idx"

    try:
        r = requests.get(idx_url, timeout=timeout, headers={"User-Agent": "nbm-subsetter/1.0"})
    except Exception as e:
        print(f"❌ Failed to fetch idx {idx_url}: {e}")
        return None

    if not r.ok or not r.text.strip():
        print(f"⚠️ Missing or empty idx: {idx_url}")
        return None

    idx_lines = r.text.strip().splitlines()
    exprs = {s: re.compile(s) for s in search_strings}
    matched_ranges = {}
    matched_vars = set()

    for n, line in enumerate(idx_lines):
        if exclude_phrase and exclude_phrase in line:
            continue
        for s, rx in exprs.items():
            if rx.search(line):
                matched_vars.add(s)
                parts = line.split(':')
                try:
                    start = int(parts[1])
                except Exception:
                    continue
                if n + 1 < len(idx_lines):
                    parts_next = idx_lines[n + 1].split(':')
                    try:
                        end = int(parts_next[1]) - 1
                        b_range = f"{start}-{end}"
                    except Exception:
                        b_range = f"{start}-"
                else:
                    b_range = f"{start}-"
                matched_ranges[b_range] = line

    if require_all_matches and len(matched_vars) != len(search_strings):
        print(f"⚠️ Not all variables matched in {remote_file}. Found: {matched_vars}")
        return None
    if not matched_ranges:
        print(f"❌ No byte ranges matched in {remote_file}")
        return None

    #ensure_parent_dir(local_filename)
    with open(local_filename, "wb") as f_out:
        for b_range in matched_ranges.keys():
            headers = {"Range": f"bytes={b_range}", "User-Agent": "nbm-subsetter/1.0"}
            try:
                rr = requests.get(remote_url, headers=headers, timeout=timeout)
            except Exception as e:
                print(f"❌ Range {b_range} failed for {remote_file}: {e}")
                return None
            if rr.status_code not in (200, 206):
                print(f"❌ HTTP {rr.status_code} on range {b_range} for {remote_file}")
                return None
            f_out.write(rr.content)

    if os.path.getsize(local_filename) > 10_000:
        print(f"✅ Downloaded [{len(matched_ranges)}] field(s) → {local_filename}")
        return local_filename
    else:
        print(f"❌ File too small or failed: {local_filename}")
        return None

init_datetime_core = build_init_datetime(init_date, init_time)
init_datetime_qmd = build_qmd_initdatetime(init_datetime_core)
init_hour_qmd = init_datetime_qmd.hour
valid_datetime = build_valid_datetime(valid_date, valid_time)
fcst_projection_core = build_forecast_projection(init_datetime_core, valid_datetime)
fcst_projection_qmd = build_forecast_projection(init_datetime_qmd, valid_datetime)

core_url = build_url(init_datetime_core, init_time, fcst_projection_core)
qmd_url = build_url(init_datetime_qmd, init_hour_qmd, fcst_projection_qmd,product="qmd")
print(f"Forecast projection for core is: {fcst_projection_core}")

print(core_url)
print(qmd_url)
print(f"Core search string is: {build_search_strings_core('WIND', fcst_projection_core)}")
core_exclude_phrase = ":ens std dev"

print(f"QMD search string is: {build_search_string_qmd('WIND', fcst_projection_qmd, percentiles)}")

local_core_filename = os.path.basename(core_url)
local_qmd_filename = os.path.basename(qmd_url)
print(f"Downloading core to: {local_core_filename}")
print(f"Downloading qmd to: {local_qmd_filename}")
core_file = download_subset(core_url, build_search_strings_core('WIND', fcst_projection_core), local_core_filename, exclude_phrase=core_exclude_phrase)
qmd_file = download_subset(qmd_url, build_search_string_qmd('WIND', fcst_projection_qmd, percentiles), local_qmd_filename)

In [ ]:
#@title ⚙️ Interpolate and create percentile grid

# Your percentile list (order matters)
percentiles = np.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100], dtype=float)

# --- 1) Read core 10 m wind (deterministic) ---
with pygrib.open(core_file) as g:
    # Select the one deterministic 10 m wind message (instantaneous at valid time)
    core_msg = next(m for m in g if m.name == "10 metre wind speed")
    core = core_msg.values  # ndarray (ny, nx); may be a masked array
    # Optionally keep lats/lons if you want to write a georeferenced output later
    # lats, lons = core_msg.latlons()

# --- 2) Read QMD percentile stack for 10 m wind ---
vals = []
with pygrib.open(qmd_file) as g:
    for m in g:
        if m.name == "10 metre wind speed" and hasattr(m, "percentileValue"):
            vals.append((float(m.percentileValue), m.values))

# Sanity: ensure we got all the percentiles you asked for
got_pcts = np.array([p for p,_ in vals], dtype=float)
#print(f"Got pcts are: {got_pcts}")
if set(got_pcts.tolist()) != set(percentiles.tolist()):
    print("Uhoh...we don't have all percentiles!")
    # You can choose to proceed with those present, or fail fast.
    # Here we proceed with the intersection, sorted.
    keep = np.isin(got_pcts, percentiles)
    vals = [v for v,k in zip(vals, keep) if k]
    got_pcts = np.array([p for p,_ in vals], dtype=float)

# Sort by percentile value to ensure monotonic index
order = np.argsort(got_pcts)
#print(f"Order is: {order}")
got_pcts = got_pcts[order]
#print(f"Sorted pcts are: {got_pcts}")
stack = np.stack([vals[i][1] for i in order], axis=0)  # shape: (nperc, ny, nx)
#print(stack)
# Make sure we’re working with ndarray (not masked) and have consistent dtype
if np.ma.isMaskedArray(core):
    core = core.filled(np.nan)
stack = np.where(np.ma.getmaskarray(stack), np.nan, stack).astype(np.float32)

# --- 3) Nearest-percentile grid ---
# For each grid point, find the QMD percentile whose value is closest to the deterministic core value
absdiff = np.abs(stack - core[np.newaxis, ...])         # (nperc, ny, nx)
idx_nearest = np.nanargmin(absdiff, axis=0)             # (ny, nx) index into got_pcts
nearest_pct = got_pcts[idx_nearest]                     # (ny, nx) nearest discrete percentile

In [ ]:
#@title ⚙️ Plot the percentile rank
#@markdown **Select your zoom or enter a custom zoom**
zoom = "Full" #@param ["SEAK", "SAK", "AncBowl", "FairbanksArea", "Custom", "Full"]
#@markdown Would you like to have a custom zoom?  If so, make sure your domain is "Custom" and enter appropriate values for lat/lons below

custom_west = -136.050976 #@param {type:"number"}
custom_north = 59.168320 #@param {type:"number"}
custom_east = -133.159775 #@param {type:"number"}
custom_south = 57.427283 #@param {type:"number"}

#@markdown Would you like to add cities to your map?
cities = False #@param {type:"boolean"}
#@markdown Thin out all cities with less than this population (to de-clutter your map) Default is 5000.
population = 5000 #@param {type:"integer"}

domain_dict = {
    "SEAK": {
        "west": -145.,
        "south": 53,
        "east": -129.,
        "north": 61.
    },
    "SAK": {
        "west": -160.,
        "south": 56.,
        "east": -140.,
        "north": 63.
    },
    "AncBowl": {
        "west": -153.,
        "south": 60.,
        "east": -144.,
        "north": 62.5
    },
    "FairbanksArea": {
        "west": -150.,
        "south": 64.,
        "east": -145.,
        "north": 65.6
    },
    "Custom": {
        "west": custom_west,
        "south": custom_south,
        "east": custom_east,
        "north": custom_north
    },
    "Full": {
        "west": -180,
        "south": 45,
        "east": -129,
        "north": 72
    }
}

if zoom in domain_dict:
  west = domain_dict[zoom]["west"]
  south = domain_dict[zoom]["south"]
  east = domain_dict[zoom]["east"]
  north = domain_dict[zoom]["north"]
else:
  print(f"Did not understand your domain selection. Plotting for full domain...")
  zoom = "full"
init_datetime_string = init_datetime_core.strftime("%Y-%m-%d %H:%M")
valid_datetime_string = valid_datetime.strftime("%Y-%m-%d %H:%M")
init_datetime_graphic = init_datetime_core.strftime("%Y%m%d%H")
valid_datetime_graphic = valid_datetime.strftime("%Y%m%d%H")

def plot_towns(ax, south, north, west, east, population=5000, resolution='10m', transform=ccrs.PlateCarree(), zorder=3):
    """
    This function will download the 'populated_places' shapefile from
    NaturalEarth, trim the shapefile based on the limits of the provided
    lat & long coords, and then plot the locations and names of the towns
    on a given GeoAxes.

    ax = a pyplot axes object
    south = south lat limit (float)
    north = north lat limit (float)
    west = west long limit (float)
    east = east long limit (float)
    resolution= str. either high res:'10m' or low res: '50m'
    population = minimum population of towns to plot (int)
    transform = a cartopy crs object
    """
    #get town locations
    shp_fn = shpreader.natural_earth(resolution=resolution, category='cultural', name='populated_places')
    shp = shpreader.Reader(shp_fn)
    xy = [pt.coords[0] for pt in shp.geometries()]
    x, y = list(zip(*xy))

    #get town names
    towns = shp.records()
    names_en = []
    max_population = []
    for town in towns:
        #print(town.attributes)
        names = town.attributes['NAME']
        pop = town.attributes['POP_MAX']
        names_en.append(names)
        max_population.append(pop)
    #print(names_en)
    #create data frame and index by the region of the plot
    all_towns = pd.DataFrame({'names_en': names_en, 'x':x, 'y':y, 'population':max_population})
    #print(all_towns.head())
    region_towns = all_towns[(all_towns.y<north) & (all_towns.y>south)
                           & (all_towns.x>west) & (all_towns.x<east)]
    region_towns = region_towns[region_towns.population > population]
    #print(region_towns.head())
    #plot the locations and labels of the towns in the region
    ax.scatter(region_towns.x.values, region_towns.y.values, c ='black', marker= '.', transform=transform, zorder=zorder)
    transform_mpl = ccrs.PlateCarree()._as_mpl_transform(ax) #this is a work-around to transform xy coords in ax.annotate
    for i, txt in enumerate(region_towns.names_en):
         ax.annotate(txt, (region_towns.x.values[i], region_towns.y.values[i]), xycoords=transform_mpl)
#--------------------------------------------------------------------
# Read the deterministic 10 m wind message to get lat/lon
with pygrib.open(core_file) as g:
    core_msg = next(m for m in g if m.name == "10 metre wind speed")
    lats, lons = core_msg.latlons()

# -------------------------------------------------------------------
# 2) Choose projection and plotting settings
#    North Polar Stereo centered over Alaska keeps the Aleutians on one map.
# -------------------------------------------------------------------
proj = ccrs.NorthPolarStereo(central_longitude=-150, true_scale_latitude=60)   # good Alaska-centric view
data_crs = ccrs.PlateCarree()                          # your lats/lons are geodetic degrees

# This span keeps the Aleutians without splitting the dateline.
extent_pc = (west, east, south, north)  # PlateCarree coords

# -------------------------------------------------------------------
# 3) Colormap: continuous for interp (0–100). If plotting nearest (0,5,...,100),
#    you can switch to a discrete colormap using BoundaryNorm (see section below).
# -------------------------------------------------------------------
# Suppose P, lats, lons already exist (your percentile grid)
percentile_bins = np.arange(0, 105, 5)      # 0–100 in steps of 5
base_cmap = plt.get_cmap("rainbow", len(percentile_bins) - 1)
cmap = ListedColormap(base_cmap(np.linspace(0, 1, len(percentile_bins) - 1)))
norm = BoundaryNorm(percentile_bins, cmap.N)

# -------------------------------------------------------------------
# 4) Plot
# -------------------------------------------------------------------
graphicname = f"{init_datetime_graphic}_NBM_In_Percentile_Valid_{valid_datetime_graphic}.png"
fig = plt.figure(figsize=(10, 9), dpi=140)
ax = plt.axes(projection=proj)
ax.set_extent(extent_pc, crs=data_crs)

# Nice map features
ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.9")
ax.add_feature(cfeature.OCEAN.with_scale("50m"))
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.5)
ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4)
ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="none", edgecolor="0.5", linewidth=0.3)
gl = ax.gridlines(crs=data_crs, draw_labels=False, linewidth=0.3, linestyle="--", alpha=0.4)

# pcolormesh wants cell corners; your lats/lons are cell centers.
# A simple approach is to drop the last row/col and let shading='auto' fill nicely.
pc = ax.pcolormesh(
    lons, lats, nearest_pct, transform=ccrs.PlateCarree(),
    cmap=cmap, norm=norm, shading="auto"
)
cb = plt.colorbar(pc, ax=ax, orientation="horizontal",
                  pad=0.03, shrink=0.9, ticks=percentile_bins)
cb.set_label("NBM 10 m Wind – Percentile Rank (%)")
if cities:
    plot_towns(ax, south, north, west, east, population=population)
plt.title(f"NBM 10 m Deterministic Wind In NBM Percentile Space\nFrom the {init_datetime_string} NBM Run Valid: {valid_datetime_string}", fontsize=12)
plt.tight_layout()
plt.savefig(graphicname, bbox_inches="tight")
print(f"Saved to: {graphicname}")

